# 📝 과제 01: 온라인 쇼핑 상담 API 이미지와 배포

<div class="day78-callout" style="margin:10px 0;padding:8px 14px;font-size:17px;line-height:1.75;color:#202124;background:#ffffff;border-left:4px solid #54769b;border-radius:4px;"><strong style="display:block;font-size:1.13em;color:#1f3a5f;margin-bottom:2px;">실습 환경</strong><strong style="font-size:1em;color:#1f3a5f;">내 PC:</strong> 저장소 파일을 고치고 commit, push 합니다(터미널). 이 노트북은 VM과 GitHub에 요청을 보내 확인하고 자가채점합니다.<br><strong style="font-size:1em;color:#1f3a5f;">GCP VM:</strong> 저장소를 clone, pull 하고 docker build, run, compose 를 실행합니다. 쇼핑 상담 API는 VM의 8081 포트를 씁니다.<br><strong style="font-size:1em;color:#1f3a5f;">GitHub:</strong> 새 공개 저장소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-deploy</code>, Actions, 이미지 레지스트리 GHCR를 씁니다.<br><strong style="font-size:1em;color:#1f3a5f;">RunPod:</strong> 교안 02에서 만들어 켜 둔 GPU 파드를 3번에서 그대로 쓰고, 교안 03과 과제 02까지 켜 둡니다(종료는 과제 02 끝).<br><strong style="font-size:1em;color:#1f3a5f;">순서:</strong> 1, 2번은 교안 01을 마친 뒤 풀고, 여기서 멈춰 교안 02로 갑니다. 3번은 교안 02를 마친 뒤 이 노트북으로 돌아와 풉니다. 3번은 교안 02가 만든 GPU 파드와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU_POD_ID</code>를 씁니다.</div>

## 만들 결과

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">FastAPI 서빙 API 단원 과제의 온라인 쇼핑 상담 API(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api</code>)를 컨테이너 이미지로 만들어 GHCR에 올리고, GCP VM에서 compose로 실행합니다. VM의 API는 교안 02의 RunPod GPU 파드 vLLM에 연결합니다.</p>
<img style="display:block;max-width:100%;height:auto;margin:0;" src="images/40_gwaje01_flow.png" width="1050" alt="내 PC에서 Dockerfile을 고쳐 push하면 GitHub Actions가 이미지를 GHCR에 올리고, GCP VM이 그 이미지를 받아 8081 포트로 실행해 RunPod GPU 파드의 vLLM을 부르는 흐름입니다.">

<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">만들 것</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Dockerfile</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.dockerignore</code>: 잠금 파일을 먼저 설치하고 root가 아닌 사용자로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api.secure_main:app</code>을 실행하는 이미지</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.github/workflows/image.yml</code>: push할 때마다 이미지를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ghcr.io/&lt;GitHub 아이디&gt;/shop-deploy</code>에 올리는 워크플로</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/vm_shop.env</code>와 VM의 compose 실행: VM 8081 포트의 쇼핑 상담 API가 GPU 파드의 vLLM으로 답하는 상태</li>
</ul>
</div>

## 준비

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">저장소 템플릿:</strong> 일차 폴더의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">배포_저장소/shop-deploy/</code>입니다. 교안 00 5-1처럼 홈 폴더(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/shop-deploy</code>)로 복사해 내 저장소로 쓰고, 파일은 VS Code의 <strong style="font-size:1.13em;color:#1f3a5f;">File &gt; Open Folder</strong>로 그 폴더를 열어 고칩니다. <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api/</code>(FastAPI 서빙 API 단원 과제의 완성 코드), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pyproject.toml</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv.lock</code>(서버 실행 라이브러리), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">deploy/compose.yaml</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tests/</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">scripts/</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows/</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">README.md</code>가 들어 있습니다. 실제 서비스 저장소에 있을 파일만 둔 것입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">워크플로 원본:</strong> 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows/</code>에는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.github/workflows/</code>로 복사해 켤 워크플로(2번의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">image.yml</code>, 과제 02의 빈칸 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ci.yml</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">cd.yml</code>)가 있습니다. GitHub은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.github/workflows/</code> 안의 워크플로만 실행하므로, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows/</code>에 둔 파일은 복사하기 전까지 실행되지 않습니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">작성할 곳:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Dockerfile</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.dockerignore</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO (n)</code> 주석 아래, 이 노트북의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO (n)</code> 셀, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;"># TODO (n)</code>이 적힌 터미널 명령 블록입니다. 요구사항 번호 (n)이 그 TODO 번호입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">자료:</strong> 3번 질문은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">data/과제/commerce_questions.jsonl</code>의 첫 질문입니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">.env:</strong> 실습자료/.env의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GITHUB_OWNER</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GITHUB_READ_TOKEN</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VM_HOST</code>를 쓰고, 3번에서는 교안 02에서 저장한 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU_POD_ID</code>도 씁니다.</li>
<li style="margin:3px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">자가채점:</strong> GitHub에 올린 파일은 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">raw.githubusercontent.com</code>에서 읽습니다. 이 주소는 몇 분 동안 옛 내용을 보여 줄 수 있으므로, push 직후 옛 내용으로 떨어지면 잠시 뒤 다시 실행합니다.</li>
</ul>
<p style="margin:6px 0;">교안 00 5-1과 같은 방법으로 템플릿 폴더를 홈 폴더로 복사해 새 공개 저장소 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-deploy</code>로 올립니다. 먼저 GitHub에서 빈 저장소를 만듭니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GitHub 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">저장소 이름 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-deploy</code>, 공개 범위 Public으로 만들고, README와 .gitignore는 추가하지 않습니다.</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> 오른쪽 위 + &gt; New repository &gt; Repository name &gt; Public &gt; Create repository</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/new</code></li>
</ul>
</div>

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">아래 블록은 템플릿 폴더를 홈 폴더로 복사하고(실습자료 안의 원본은 그대로 둡니다), 그 폴더를 새 저장소로 올립니다. <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">&lt;GitHub 아이디&gt;</code>를 내 GitHub 아이디(.env의 <code style="color:inherit;background:#f3f6fa;font-size:0.95em;">GITHUB_OWNER</code>)로 바꾼 뒤 실행합니다.</strong> 바꾸지 않으면 터미널이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&gt;</code>를 기호로 읽어 오류가 납니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
cp -r day78_모델배포_Docker_CICD/배포_저장소/shop-deploy ~/shop-deploy
pushd ~/shop-deploy
git init -b main
git add .
git commit -m "쇼핑 상담 API 배포 저장소 템플릿"
git remote add origin https://github.com/<GitHub 아이디>/shop-deploy.git
git push -u origin main
popd
```

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
cd ~
git clone https://github.com/<GitHub 아이디>/shop-deploy.git
cd ~/shop-deploy
pwd   # 끝이 shop-deploy이면 맞습니다
```

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">&lt;GitHub 아이디&gt;</code>는 내 GitHub 아이디로 바꿉니다. 공개 저장소이므로 VM에서는 로그인 없이 받습니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# ---- 라이브러리, 경로, 주소 ----
import json
import os
from pathlib import Path

import httpx
import pandas as pd
from dotenv import load_dotenv

day_dir = Path(".").resolve()       # data, images, output 이 있는 일차 폴더(day78_모델배포_Docker_CICD)
data_dir = day_dir / "data"
output_dir = day_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 키와 주소는 실습자료/.env 에 모아 둡니다(화면에 출력하지 않습니다)
env_path = day_dir.parent / ".env"
load_dotenv(env_path, override=True)

SERVED_NAME = "qwen3.5-4b"            # vLLM의 --served-model-name

# 테이블의 긴 글(질문, 답)이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


# 학생 저장소(GitHub API, 올린 파일 원문)와 VM 의 쇼핑 상담 API 주소
GITHUB_OWNER = os.environ["GITHUB_OWNER"]
# GitHub API 는 로그인 없이 부르면 IP 하나에 시간당 60번이라 같은 와이파이의 교실 전체가 나눠 씁니다. 읽기 전용 토큰을 붙이면 학생마다 시간당 5000번입니다
GITHUB_HEADERS = {"Authorization": "Bearer " + os.environ["GITHUB_READ_TOKEN"], "Accept": "application/vnd.github+json"}  # 교안 00 5-2 의 읽기 전용 토큰(값은 출력하지 않습니다)
REPO_API = "https://api.github.com/repos/" + GITHUB_OWNER + "/shop-deploy"
RAW_URL = "https://raw.githubusercontent.com/" + GITHUB_OWNER + "/shop-deploy/main"
SHOP_URL = "http://" + os.environ["VM_HOST"] + ":8081"

# shop_api 의 pro 요금제 연습용 서비스 키(60초에 30번, 이미지 안의 service_keys.json 에 들어 있습니다)
SHOP_PRO_KEY = "shop-pro-demo-0002"

# 3번에서 보낼 쇼핑 질문(첫 질문 하나)
commerce_questions = load_jsonl(data_dir / "과제" / "commerce_questions.jsonl")
sample_question = commerce_questions[0]["question"]
print("질문 수:", len(commerce_questions), "| 3번 질문:", sample_question)

## 1. 쇼핑 상담 API 이미지의 Dockerfile

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배경:</strong> 쇼핑 상담 API를 어느 컴퓨터에서나 같은 라이브러리와 실행 명령으로 띄우려고 이미지로 만듭니다. 코드만 고쳤을 때 라이브러리 설치를 다시 하지 않도록 잠금 파일을 먼저 설치하고, 키 파일과 개발용 파일은 이미지에 넣지 않습니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong> (<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">Dockerfile</code>의 TODO (1)~(4), <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.dockerignore</code>의 TODO (5))</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) 베이스 이미지는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">python:3.12-slim</code>입니다. uv 0.13.7의 실행 파일 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uvx</code>를 uv 공식 이미지 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ghcr.io/astral-sh/uv:0.13.7</code>에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/bin/</code>으로 복사하고, 작업 폴더를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/app</code>으로 정합니다.</li>
<li style="margin:3px 0;">(2) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">pyproject.toml</code>과 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv.lock</code> 두 파일만 한 줄의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">COPY</code>로 먼저 복사하고, 그 다음 줄에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uv sync</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--locked</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--no-dev</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">--no-cache</code> 세 옵션으로 실행합니다.</li>
<li style="margin:3px 0;">(3) 서버 코드 폴더 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api</code>를 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/app/shop_api</code>로 복사합니다. 이 줄은 (2)의 두 줄보다 뒤에 둡니다.</li>
<li style="margin:3px 0;">(4) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">CMD</code>를 JSON 배열 모양(exec 형식)으로 써서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">uvicorn</code>이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop_api.secure_main:app</code>을 호스트 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">0.0.0.0</code>, 포트 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">8080</code>으로 실행하게 합니다.</li>
<li style="margin:3px 0;">(5) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.dockerignore</code>에 한 줄에 하나씩 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.git</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.venv</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">deploy/.env</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">**/__pycache__</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">tests</code>, <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">deploy</code> 일곱 줄을 적습니다.</li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · shop-deploy/Dockerfile</strong></div>

```dockerfile
# 온라인 쇼핑 상담 API 이미지(과제 01)

# TODO (1): 파이썬 3.12 slim 버전을 베이스로 정하고, uv 0.13.7 실행 파일을 /bin/ 에 복사하고, 작업 폴더를 /app 으로 정하세요

# TODO (2): pyproject.toml 과 uv.lock 만 먼저 복사하고, 잠금 파일 그대로(--locked) 테스트 도구 없이(--no-dev) 캐시를 남기지 않고(--no-cache) 설치하세요

# TODO (3): 서버 코드 폴더 shop_api 를 /app/shop_api 로 복사하세요

# 가상환경의 실행 파일을 먼저 찾게 하고, 쓰는 포트를 적어 둔다(제공)
ENV PATH="/app/.venv/bin:$PATH"
EXPOSE 8080

# TODO (4): uvicorn 으로 shop_api.secure_main:app 을 0.0.0.0 의 8080 포트에서 실행하는 명령을 exec 형식으로 쓰세요
```

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · shop-deploy/.dockerignore</strong></div>

```text
# 이미지에 넣지 않을 것(과제 01). 경로는 저장소 맨 위 기준이고, 아래 폴더까지 빼려면 **/ 를 붙인다
# 워크플로 원본 폴더는 처음부터 적어 두었다
workflows
# TODO (5): 저장소 기록, 가상환경, 키 파일 두 개, 파이썬 캐시(아래 폴더까지), 테스트, 배포 설정 폴더를 한 줄에 하나씩 적으세요
```

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 교안 00의 3-2(Dockerfile 명령)와 교안 01의 2절(서빙 API Dockerfile, .dockerignore)을 참고합니다.
세부구현
1) 다른 이미지 안의 파일은 COPY 명령의 --from 옵션에 그 이미지 이름을 주어 가져옵니다.
2) 잠금 파일 두 개를 먼저 복사하고 RUN으로 설치하면, 코드만 바뀐 재빌드에서 설치 층이 캐시로 남습니다.
3) exec 형식의 CMD는 실행 파일과 인자를 큰따옴표 글자로 하나씩 나눠 대괄호 안에 씁니다.
4) 컨테이너 밖에서 포트 연결로 닿으려면 서버가 모든 주소(0.0.0.0)에서 기다려야 합니다.
```

</details>

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">두 파일을 저장한 뒤 내 PC 터미널에서 commit, push 합니다. 이미지 빌드는 2번에서 GitHub Actions가 합니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
pushd ~/shop-deploy
git add Dockerfile .dockerignore
git commit -m "쇼핑 상담 API Dockerfile 과 .dockerignore 작성"
git push
popd
```

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# [자가채점] GitHub 에 올린 Dockerfile 과 .dockerignore 를 확인합니다
dockerfile_response = httpx.get(RAW_URL + "/Dockerfile", timeout=30)
assert dockerfile_response.status_code == 200, f"올린 Dockerfile 을 읽지 못했습니다(상태 코드 {dockerfile_response.status_code}). 저장소 이름이 shop-deploy 이고 Public 인지, main 에 push 했는지 확인하세요."

# 주석과 빈 줄을 뺀 명령 줄만 남깁니다
dockerfile_lines = []
for line in dockerfile_response.text.splitlines():
    stripped = line.strip()
    if stripped and not stripped.startswith("#"):
        dockerfile_lines.append(stripped)


# 조건에 맞는 첫 줄의 번호를 찾습니다(없으면 -1)
def find_line(prefix, word=""):
    for index, line in enumerate(dockerfile_lines):
        if line.startswith(prefix) and word in line:
            return index
    return -1


# (1) 베이스, uv 실행 파일, 작업 폴더
assert find_line("FROM python:3.12-slim") == 0, "첫 명령이 FROM python:3.12-slim 이 아닙니다(TODO (1)). 파일을 고쳤는데도 이 메시지가 나오면 push 했는지 보고, push 직후라면 몇 분 뒤 다시 실행하세요."
uv_line = find_line("COPY --from=ghcr.io/astral-sh/uv:0.13.7")
assert uv_line > 0 and "/bin/" in dockerfile_lines[uv_line], "uv 공식 이미지에서 uv 실행 파일을 /bin/ 으로 복사하는 COPY --from 줄이 없습니다(TODO (1))."
assert find_line("WORKDIR /app") > 0, "작업 폴더를 /app 으로 정하는 WORKDIR 줄이 없습니다(TODO (1))."

# (2), (3) 잠금 파일 먼저, 설치, 그 다음 코드
lock_line = find_line("COPY", "uv.lock")
sync_line = find_line("RUN", "uv sync")
code_line = find_line("COPY", "shop_api")
assert lock_line > 0 and "pyproject.toml" in dockerfile_lines[lock_line], "pyproject.toml 과 uv.lock 을 한 줄로 복사하는 COPY 줄이 없습니다(TODO (2))."
assert sync_line > lock_line, "uv sync 를 실행하는 RUN 줄이 잠금 파일 COPY 줄 뒤에 없습니다(TODO (2))."
for option in ["--locked", "--no-dev", "--no-cache"]:
    assert option in dockerfile_lines[sync_line].split(), f"uv sync 줄에 {option} 옵션이 없습니다(TODO (2))."
assert code_line > sync_line, "COPY shop_api 줄이 없거나 uv sync 줄보다 앞에 있습니다. 코드는 설치 뒤에 복사해야 코드만 고친 재빌드에서 설치 층이 캐시로 남습니다(TODO (3))."
assert dockerfile_lines[code_line].split()[-1].rstrip("/") in ["shop_api", "./shop_api", "/app/shop_api"], f"{dockerfile_lines[code_line]!r} 의 복사 위치가 /app/shop_api 가 아닙니다(TODO (3))."

# (4) exec 형식의 CMD
cmd_line = find_line("CMD")
assert cmd_line > 0 and dockerfile_lines[cmd_line].startswith("CMD ["), "CMD 줄이 없거나 JSON 배열 모양(exec 형식)이 아닙니다(TODO (4))."
cmd_parts = json.loads(dockerfile_lines[cmd_line][len("CMD "):])
for part in ["uvicorn", "shop_api.secure_main:app", "0.0.0.0", "8080"]:
    assert part in cmd_parts, f"CMD 의 배열 {cmd_parts} 에 {part!r} 가 없습니다(TODO (4))."

# (5) .dockerignore 일곱 줄
dockerignore_response = httpx.get(RAW_URL + "/.dockerignore", timeout=30)
assert dockerignore_response.status_code == 200, f"올린 .dockerignore 를 읽지 못했습니다(상태 코드 {dockerignore_response.status_code})."
ignored = []
for line in dockerignore_response.text.splitlines():
    stripped = line.strip().rstrip("/")
    if stripped and not stripped.startswith("#"):
        ignored.append(stripped)
for name in [".git", ".venv", ".env", "deploy/.env", "**/__pycache__", "tests", "deploy"]:
    assert name in ignored, f".dockerignore 에 {name} 줄이 없습니다(TODO (5))."
print("✅ 통과!")

## 2. GitHub Actions로 GHCR에 이미지 올리기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배경:</strong> 배포할 VM이 저장소를 빌드하지 않고 이미 만든 이미지를 받아 실행하도록, push할 때마다 GitHub Actions가 이미지를 빌드해 GHCR(GitHub Container Registry)에 올리게 합니다. VM이 로그인 없이 받을 수 있게 패키지는 공개로 둡니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) 내 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">workflows/image.yml</code>(제공)을 같은 저장소의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.github/workflows/image.yml</code>로 복사해 commit, push 합니다. 그 push로 시작된 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">image</code> 워크플로 실행이 성공해 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ghcr.io/&lt;GitHub 아이디 소문자&gt;/shop-deploy</code>에 커밋 sha 태그와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code> 태그가 올라갑니다.</li>
<li style="margin:3px 0;">(2) GitHub의 패키지 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-deploy</code> 공개 범위가 Public입니다(Private이면 Public으로 바꿉니다). 로그인하지 않은 요청으로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">latest</code> 이미지 정보를 받을 수 있어야 합니다.</li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · shop-deploy/workflows/image.yml(제공)</strong></div>

```yaml
# 이미지를 빌드해 GHCR 에 올리는 워크플로(과제 01). 이 파일을 저장소의 .github/workflows/ 로 복사하면 GitHub 이 실행한다(workflows/ 에 있는 동안은 실행되지 않는다)
name: image

on:
  push:
    branches: [main]
  workflow_dispatch:

permissions:
  contents: read
  packages: write                                   # GITHUB_TOKEN 에 GHCR 쓰기 권한을 준다(없으면 올리는 단계에서 거절)

jobs:
  build:
    runs-on: ubuntu-latest
    timeout-minutes: 15
    steps:
      - name: 저장소 받기
        uses: actions/checkout@v7
      - name: 이미지 이름 정하기
        id: image
        # 이미지 이름은 소문자만 된다. ${GITHUB_REPOSITORY,,} 는 "계정/저장소"를 소문자로 바꾼다
        run: echo "name=ghcr.io/${GITHUB_REPOSITORY,,}" >> "$GITHUB_OUTPUT"
      # Docker 의 확장 빌드 도구 buildx. 아래 type=gha 캐시는 buildx 로 빌드할 때만 쓸 수 있다
      - name: 빌드 도구(buildx) 준비
        uses: docker/setup-buildx-action@v4
      # docker login 과 같다. 실행한 사람(github.actor)과 GITHUB_TOKEN 으로 ghcr.io 에 로그인한다
      - name: GHCR 로그인
        uses: docker/login-action@v4
        with:
          registry: ghcr.io
          username: ${{ github.actor }}
          password: ${{ secrets.GITHUB_TOKEN }}     # 실행마다 GitHub 가 주는 토큰(따로 등록하지 않음)
      - name: 빌드하고 올리기
        uses: docker/build-push-action@v7
        with:
          context: .                                # 저장소 폴더를 빌드한다
          push: true                                # tags 의 이름을 모두 GHCR 에 올린다
          # 태그 둘: 커밋 sha(github.sha, 이 실행을 시작한 커밋. 되돌릴 때 부르는 이름)와 latest(가장 최근)
          tags: |
            ${{ steps.image.outputs.name }}:${{ github.sha }}
            ${{ steps.image.outputs.name }}:latest
          cache-from: type=gha                      # 지난 실행의 레이어 캐시를 가져온다
          cache-to: type=gha,mode=max               # 이번 실행의 모든 레이어를 캐시로 남긴다
```

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 교안 01의 4절(이미지 이름과 태그), 5절(GitHub Actions로 이미지 올리기)을 참고합니다.
세부구현
1) GitHub Actions는 저장소의 .github/workflows 폴더에 있는 워크플로 파일만 실행합니다. 폴더가 없으면 먼저 만듭니다.
2) 복사한 파일을 git add로 올린 뒤 commit, push하면 main의 push가 워크플로를 시작합니다.
3) 실행 결과는 저장소의 Actions 탭, 공개 범위는 패키지 설정 화면의 Danger Zone에서 봅니다.
4) pushd 로 내 저장소에 들어가 복사하고 커밋한 뒤 popd 로 돌아옵니다(교안 01 5-2와 같은 순서, Windows의 폴더 만들기도 같음).
```

</details>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
# TODO (1): 내 저장소의 workflows/image.yml 을 .github/workflows/image.yml 로 복사하고, commit 해 push 하세요
```

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">GitHub 화면에서 할 일</strong></div>
<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">Actions 탭에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">image</code> 실행이 초록 체크로 끝났는지 보고, 패키지 설정에서 공개 범위를 확인합니다(요구사항 (2)).</p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> 저장소 위쪽 Actions &gt; image &gt; 맨 위 실행</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/&lt;GitHub 아이디&gt;/shop-deploy/actions</code></li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">누를 곳:</strong> 내 프로필 &gt; Packages &gt; shop-deploy &gt; Package settings &gt; Danger Zone &gt; Change visibility &gt; Public</li>
<li style="margin:3px 0;"><strong class="day78-item" style="font-size:1em;color:#1f3a5f;">주소:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://github.com/users/&lt;GitHub 아이디&gt;/packages/container/shop-deploy/settings</code></li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# [자가채점] image 워크플로의 성공한 실행과 로그인 없이 받는 GHCR 이미지 정보를 확인합니다
image_runs = httpx.get(REPO_API + "/actions/workflows/image.yml/runs", params={"status": "success", "per_page": 5}, headers=GITHUB_HEADERS, timeout=30)
assert image_runs.status_code == 200, f"image 워크플로를 찾지 못했습니다(상태 코드 {image_runs.status_code}). .github/workflows/image.yml 로 복사해 push 했는지 확인하세요(TODO (1))."
assert image_runs.json()["total_count"] >= 1, "성공한 image 실행이 없습니다. Actions 탭에서 실행이 끝났는지, 실패했다면 어느 단계의 로그에서 멈췄는지 확인하세요(TODO (1))."

# (2) GHCR 의 로그인하지 않은 사용자용 토큰으로 latest 이미지 정보(manifest)를 받습니다
image_path = GITHUB_OWNER.lower() + "/shop-deploy"
token_response = httpx.get("https://ghcr.io/token", params={"scope": "repository:" + image_path + ":pull"}, timeout=30)
assert token_response.status_code == 200, f"GHCR 이 받기 토큰을 주지 않았습니다(상태 코드 {token_response.status_code}). 패키지가 Public 인지 확인하세요(요구사항 (2))."
manifest_types = "application/vnd.oci.image.index.v1+json, application/vnd.oci.image.manifest.v1+json, application/vnd.docker.distribution.manifest.list.v2+json, application/vnd.docker.distribution.manifest.v2+json"
manifest = httpx.get(
    "https://ghcr.io/v2/" + image_path + "/manifests/latest",
    headers={"Authorization": "Bearer " + token_response.json()["token"], "Accept": manifest_types},
    timeout=30,
)
assert manifest.status_code == 200, f"로그인 없이 ghcr.io/{image_path}:latest 정보를 받지 못했습니다(상태 코드 {manifest.status_code}). 패키지 공개 범위를 Public 으로 바꿨는지 확인하세요(요구사항 (2))."
print("✅ 통과!")

## 3. VM에서 compose로 실행하고 GPU 파드에 연결하기

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">교안 02를 마친 뒤 풉니다.</strong> 교안 02 3절에서 만든 GPU 파드(<code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">serving-vllm</code>)와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">.env</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU_POD_ID</code>, 교안 02 1-2에서 만든 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>가 있어야 합니다. 노트북을 새로 열었다면 맨 위 준비 셀부터 다시 실행합니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">배경:</strong> 이미지 안에는 키와 주소를 넣지 않았으므로, VM의 설정 파일로 GHCR 이미지와 GPU 파드의 vLLM 주소, 키를 넘겨 실제 상담을 하는 API로 띄웁니다. 교안의 서빙 API(8080)와 겹치지 않게 VM의 8081 포트를 씁니다.</p>
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">요구사항</strong></p>
<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;">(1) <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/vm_shop.env</code> 파일에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">이름=값</code> 한 줄씩 네 줄을 이 순서로 저장합니다. 값은 화면에 출력하지 않습니다.<ul style="margin:8px 0;padding-left:25px;">
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SHOP_IMAGE</code>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">ghcr.io/</code> 뒤에 소문자로 바꾼 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GITHUB_OWNER</code>, 그 뒤에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/shop-deploy</code></li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_BASE_URL</code>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">https://</code> 뒤에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">GPU_POD_ID</code>, 그 뒤에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">-8000.proxy.runpod.net/v1</code></li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code>: .env의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_API_KEY</code> 값</li>
<li style="margin:3px 0;"><code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">VLLM_TIMEOUT_SECONDS</code>: <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">90</code></li>
</ul>
</li>
<li style="margin:3px 0;">(2) 내 PC 터미널에서 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">output/vm_shop.env</code>를 VM의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/shop-deploy/deploy/.env</code>로 올립니다. VM에서는 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/shop-deploy/deploy</code> 폴더에서 compose로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-api</code> 서비스를 백그라운드로 띄운 뒤 실행 상태를 확인합니다.</li>
<li style="margin:3px 0;">(3) <strong style="font-size:1.13em;color:#1f3a5f;"><code style="color:inherit;background:#f3f6fa;font-size:0.95em;">shop_answer</code>:</strong> <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SHOP_URL</code>의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/shop/chat</code>에 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">X-API-Key</code> 헤더로 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">SHOP_PRO_KEY</code>를 주고 본문 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">{&quot;question&quot;: sample_question}</code>을 POST로 보낸 응답 JSON(딕셔너리)입니다. 요청의 제한 시간은 120초로 줍니다(VM의 vLLM 제한 시간 90초보다 길게).</li>
</ul>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">파일 · shop-deploy/deploy/compose.yaml(제공)</strong></div>

```yaml
# GCP VM 에서 온라인 쇼핑 상담 API 컨테이너를 띄우는 실행 설정(과제 01)
# 실행: 이 deploy 폴더에서 docker compose up -d. 교안의 서빙 API(8080)와 겹치지 않게 VM 의 8081 포트를 쓴다
# 프로젝트 이름. 정하지 않으면 폴더 이름(deploy)이 되어, 같은 VM 의 다른 저장소 compose 와 겹친다
name: shop-deploy

services:
  shop-api:
    image: ${SHOP_IMAGE}:${IMAGE_TAG:-latest}
    ports:
      - "8081:8080"
    env_file: .env
    restart: unless-stopped
```

<details style="margin:8px 0;"><summary style="font-size:17px;color:#1f3a5f;font-weight:600;cursor:pointer;margin:6px 0;">힌트</summary>

```text
접근방법: 교안 02의 4절(VM에서 compose로 서빙 API 실행), 5절(PC에서 VM 호출)을 참고합니다.
세부구현
1) 이미지 이름은 소문자만 되므로 GitHub 아이디에 문자열의 lower 메서드를 씁니다.
2) 네 줄을 리스트에 담고 줄바꿈 글자로 이어 Path의 write_text로 저장합니다. 화면에는 키 이름만 보입니다.
3) scp의 받는 쪽은 부록 01(GCP VM 준비)에서 ~/.ssh/config에 적은 Host 이름 뒤에 콜론과 VM 안의 경로를 붙입니다.
4) compose는 compose.yaml이 있는 폴더에서 up 명령에 백그라운드 옵션을 주고, ps 명령으로 상태를 봅니다. compose는 같은 폴더의 .env에서 이미지 이름도 읽습니다.
5) 상담 요청은 httpx의 post에 headers, json, timeout 키워드 인자를 주고, 상태 코드를 확인한 뒤 응답의 json 메서드로 딕셔너리를 꺼냅니다.
```

</details>

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GPU 파드 확인(제공):</strong> 교안 02에서 만들어 켜 둔 GPU 파드의 vLLM이 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">/v1/models</code>에 200으로 답하는지 확인합니다. 켜져 있는 동안 시간당 요금이 나갑니다. 파드가 없거나 200이 오지 않으면 교안 02 3절의 설정으로 새 파드를 만들고(요청을 받기까지 약 10분), 교안 02 7-2(&quot;새 GPU 파드로 바꿀 때 VM의 설정 파일 고치기&quot;)로 서빙 API 설정을 바꾼 뒤 이 셀을 다시 실행합니다. 쇼핑 상담 API 설정 파일은 아래 TODO (1), (2)가 새 파드 id로 만듭니다.</p>
</div>

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
import time

# 새 파드를 만들어 .env 의 GPU_POD_ID 를 바꿨을 수 있으므로 .env 를 다시 읽습니다
load_dotenv(env_path, override=True)

# vLLM 이 모델을 올려 /v1/models 가 200 이 될 때까지 10초마다 확인합니다(최대 60번, 약 10분)
vllm_models_url = "https://" + os.environ["GPU_POD_ID"] + "-8000.proxy.runpod.net/v1/models"
vllm_headers = {"Authorization": "Bearer " + os.environ["VLLM_API_KEY"]}  # vLLM 은 키를 Authorization 헤더에 'Bearer 키' 모양으로 받습니다
vllm_status = None
for attempt in range(60):
    vllm_status = httpx.get(vllm_models_url, headers=vllm_headers, timeout=30).status_code
    if vllm_status == 200:
        break
    time.sleep(10)
print("vLLM /v1/models 상태 코드:", vllm_status)

In [ ]:
# TODO (1): VM 의 deploy/.env 가 될 네 줄을 env_lines 에 담고 output/vm_shop.env 에 저장하세요(값은 출력하지 않습니다)
vm_shop_env = output_dir / "vm_shop.env"
env_lines = ...


# 키 이름만 출력합니다(값에는 vLLM 키가 들어 있습니다)
env_keys = [line.split("=")[0] for line in env_lines]
print("저장:", vm_shop_env.name, env_keys)

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · 내 PC 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더에서 연 터미널</span></div>

```bash
# TODO (2): output/vm_shop.env 를 VM 의 ~/shop-deploy/deploy/.env 로 올리세요
```

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">복사해서 실행 · GCP VM 터미널</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">VS Code Remote-SSH로 연 VM 창의 터미널</span></div>

```bash
# TODO (2): ~/shop-deploy/deploy 폴더에서 compose 로 shop-api 를 백그라운드로 띄운 뒤 상태를 확인하세요
```

<div class="day78-label" style="margin:14px 0 4px;padding:5px 12px;font-size:16px;line-height:1.6;color:#1f3a5f;background:#eef3f8;border-left:4px solid #54769b;border-radius:3px;"><strong style="color:#1f3a5f;">셀 실행 · 내 PC 노트북</strong> <span style="color:#5f6368;font-size:16px;font-weight:normal;">실습자료 폴더를 연 내 PC의 VS Code에서 실행합니다</span></div>

In [ ]:
# TODO (3): pro 키로 쇼핑 질문 하나를 VM 의 상담 주소에 보내고 응답 JSON 을 shop_answer 에 담으세요
shop_answer = ...

display(shop_answer)

In [ ]:
# [자가채점] VM 설정 파일, compose 로 띄운 API, GPU 파드를 거친 상담 응답을 확인합니다(키 값은 출력하지 않습니다)
# (1) output/vm_shop.env 의 네 줄
assert vm_shop_env.exists(), "output/vm_shop.env 가 없습니다. TODO (1) 셀에서 파일을 저장하세요."
saved = {}
saved_keys = []
for line in vm_shop_env.read_text(encoding="utf-8").splitlines():
    if line.strip():
        name, _, value = line.partition("=")
        saved[name] = value
        saved_keys.append(name)
assert saved_keys == ["SHOP_IMAGE", "VLLM_BASE_URL", "VLLM_API_KEY", "VLLM_TIMEOUT_SECONDS"], f"vm_shop.env 의 키가 {saved_keys}입니다. 네 키를 요구사항의 순서로 쓰세요(TODO (1))."
assert saved["SHOP_IMAGE"] == "ghcr.io/" + GITHUB_OWNER.lower() + "/shop-deploy", f"SHOP_IMAGE 가 {saved['SHOP_IMAGE']!r}입니다. ghcr.io/<소문자 GitHub 아이디>/shop-deploy 여야 합니다(TODO (1))."
assert saved["VLLM_BASE_URL"] == "https://" + os.environ["GPU_POD_ID"] + "-8000.proxy.runpod.net/v1", "VLLM_BASE_URL 이 GPU 파드의 proxy 주소(끝에 /v1)가 아닙니다(TODO (1))."
assert saved["VLLM_API_KEY"] == os.environ["VLLM_API_KEY"], "VLLM_API_KEY 가 .env 의 값과 다릅니다(TODO (1))."
assert saved["VLLM_TIMEOUT_SECONDS"] == "90", f"VLLM_TIMEOUT_SECONDS 가 {saved['VLLM_TIMEOUT_SECONDS']!r}입니다. RunPod proxy 의 100초 제한보다 짧은 90 이어야 합니다(TODO (1))."

# (2) VM 의 compose 서비스: 상태 확인 200, 앱은 secure_main(1번 CMD), 키 없는 상담 401
health = httpx.get(SHOP_URL + "/healthz", timeout=30)
assert health.status_code == 200, f"VM 의 /healthz 가 {health.status_code}입니다. VM 의 deploy 폴더에서 docker compose ps 와 docker compose logs 로 shop-api 상태를 확인하세요(TODO (2))."
openapi = httpx.get(SHOP_URL + "/openapi.json", timeout=30).json()
assert openapi["info"]["title"] == "온라인 쇼핑 상담 API(서비스 키 필요)", f"VM 에서 도는 앱의 제목이 {openapi['info']['title']!r}입니다. 1번 Dockerfile 의 CMD 가 shop_api.secure_main:app 을 실행하는지 확인하세요(1번 TODO (5))."
no_key = httpx.post(SHOP_URL + "/v1/shop/chat", json={"question": sample_question}, timeout=30)
assert no_key.status_code == 401 and no_key.json()["error"]["code"] == "invalid_service_key", f"키 없는 상담 요청이 {no_key.status_code}입니다. 401 과 invalid_service_key 여야 합니다."

# (3) GPU 파드의 vLLM 이 만든 답
assert isinstance(shop_answer, dict), f"shop_answer 의 자료형이 {type(shop_answer).__name__}입니다. 응답의 json 메서드로 꺼낸 딕셔너리를 담으세요(TODO (3))."
assert isinstance(shop_answer["answer"], str) and len(shop_answer["answer"]) > 0, "shop_answer 의 answer 가 비어 있습니다. 상담 응답 전체를 담았는지 확인하세요(TODO (3))."
assert isinstance(shop_answer["usage"], dict) and shop_answer["usage"]["total_tokens"] > 0, "shop_answer 의 usage 에 토큰 수가 없습니다(TODO (3))."
assert shop_answer["model"] == SERVED_NAME, f"답한 모델이 {shop_answer['model']!r}입니다. VM 이 GPU 파드의 vLLM({SERVED_NAME})에 연결됐는지 확인하세요."
print("✅ 통과!")

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;"><strong style="font-size:1.13em;color:#1f3a5f;">GPU 파드는 켜 둡니다.</strong> GPU 파드와 VM의 두 API(서빙 API, 쇼핑 상담 API)는 교안 03과 과제 02에서도 쓰므로 그대로 두고, GPU 파드는 과제 02 끝에서 종료합니다.</p>
</div>

## 마무리

<div class="day78-explanation" style="font-size:17px;line-height:1.75;color:#202124;background:#ffffff;margin:4px 0;padding:0;">
<p style="margin:6px 0;">VM의 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">shop-api</code> 컨테이너(8081 포트)와 <code style="color:#24292e;background:#f3f6fa;font-size:0.95em;">~/shop-deploy/deploy/.env</code>는 과제 02에서 GitHub Actions가 새 이미지를 배포하는 대상이므로 지우지 않습니다.</p>
</div>